# 02 — Metadata Preparation

## Why metadata matters

A vibration recording tells us what the sensor measured. **Metadata tells us where that recording came from.** For this project, we need to retain the identity of the physical bearing, its condition, the operating condition and the measurement number.

This is essential for V2: when we later create training, validation and test sets, every recording from the same physical bearing must stay in one partition. If we discard the bearing ID now, we make that evaluation much harder to do correctly.

## Two label views from the same bearing

The notebook maps each bearing ID to two labels:

| Bearing condition | `fault_class` | `binary_label` |
|---|---:|---:|
| Healthy | 0 | 0 |
| Outer | 1 | 1 |
| Inner | 2 | 1 |
| Both | 3 | 1 |

- `fault_location` preserves the four detailed categories: healthy, outer, inner and both.
- `fault_class` stores those categories as numeric class IDs.
- `binary_label` supports the simpler Healthy-versus-Damaged task.

This is how the project can study both Stage 1 (binary detection) and Stage 2 (fault-type classification) without losing the original category.

## Reading the filename

A filename such as `N15_M07_F10_KA01_12.mat` carries several pieces of identity:

- `N15_M07_F10`: operating-condition identifier
- `KA01`: physical bearing ID
- `12`: measurement number
- `.mat`: MATLAB file format

The notebook checks the four expected operating-condition groups and expects 20 files for each bearing-condition combination. The bearing metadata is saved locally as `data/processed/bearing_metadata.csv`.

## Output

The output is one row per physical bearing, with its bearing ID and label mapping. The per-recording file inventory is checked separately through the bearing and operating-condition folders.

Start Jupyter with its working directory set to the `notebooks` folder so the relative paths to `../data/raw` and `../data/processed` resolve correctly. Raw recordings are not committed to Git.

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# STAGE 2: MASTER BEARING METADATA
# ============================================================

RAW_DIR = Path("../data/raw")

# ------------------------------------------------------------
# Bearing categories from Paderborn documentation
# ------------------------------------------------------------

healthy_bearings = [
    "K001", "K002", "K003", "K004", "K005", "K006"
]

outer_bearings = [
    "KA01", "KA03", "KA04", "KA05", "KA06",
    "KA07", "KA08", "KA09", "KA15", "KA16",
    "KA22", "KA30"
]

inner_bearings = [
    "KI01", "KI03", "KI04", "KI05", "KI07",
    "KI08", "KI14", "KI16", "KI17", "KI18", "KI21"
]

both_bearings = [
    "KB23", "KB24", "KB27"
]

expected_bearings = (
    healthy_bearings
    + outer_bearings
    + inner_bearings
    + both_bearings
)

print("Total expected bearings:", len(expected_bearings))


# ------------------------------------------------------------
# Create metadata
# ------------------------------------------------------------

metadata_records = []

for bearing in expected_bearings:

    if bearing in healthy_bearings:
        fault_location = "healthy"
        fault_class = 0
        binary_label = 0

    elif bearing in outer_bearings:
        fault_location = "outer"
        fault_class = 1
        binary_label = 1

    elif bearing in inner_bearings:
        fault_location = "inner"
        fault_class = 2
        binary_label = 1

    elif bearing in both_bearings:
        fault_location = "both"
        fault_class = 3
        binary_label = 1

    metadata_records.append({
        "bearing": bearing,
        "fault_location": fault_location,
        "fault_class": fault_class,
        "binary_label": binary_label
    })

bearing_metadata = pd.DataFrame(metadata_records)

display(bearing_metadata)

In [ ]:
# ============================================================
# VERIFY CLASS DISTRIBUTION
# ============================================================

print("Fault-location distribution:")
print(
    bearing_metadata["fault_location"].value_counts()
)

print("\nFault-class distribution:")
print(
    bearing_metadata["fault_class"].value_counts().sort_index()
)

print("\nBinary-label distribution:")
print(
    bearing_metadata["binary_label"].value_counts().sort_index()
)

print("\nTotal bearings:", len(bearing_metadata))

In [ ]:
# ============================================================
# VERIFY OPERATING-CONDITION FILE COUNTS
# ============================================================

conditions = [
    "N15_M07_F10",
    "N09_M07_F10",
    "N15_M01_F10",
    "N15_M07_F04"
]

condition_records = []

for bearing in expected_bearings:

    bearing_dir = RAW_DIR / bearing

    for condition in conditions:

        matching_files = list(
            bearing_dir.glob(
                f"{condition}_{bearing}_*.mat"
            )
        )

        condition_records.append({
            "bearing": bearing,
            "condition": condition,
            "file_count": len(matching_files)
        })

condition_df = pd.DataFrame(condition_records)

display(condition_df)

In [ ]:
# ============================================================
# CONDITION VERIFICATION SUMMARY
# ============================================================

print(
    "Total bearing-condition combinations:",
    len(condition_df)
)

print("\nFile-count distribution:")
print(
    condition_df["file_count"]
    .value_counts()
    .sort_index()
)

print("\nAny bearing-condition group with != 20 files:")

display(
    condition_df[
        condition_df["file_count"] != 20
    ]
)

In [ ]:
# ============================================================
# SAVE MASTER METADATA
# ============================================================

from pathlib import Path

# Create processed directory if it does not exist
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

metadata_path = PROCESSED_DIR / "bearing_metadata.csv"

bearing_metadata.to_csv(
    metadata_path,
    index=False
)

print("Saved:", metadata_path.resolve())